# Control de Calidad de Anotaciones Multiclase (CRISP-DM Fase 3)

Antes de proceder con el modelado en YOLO, es mandatorio auditar rigurosamente las anotaciones ground truth para garantizar:
1. **Formato YOLO estricto:** 5 columnas por línea (`class_id x_center y_center width height`).
2. **Normalización en $[0.0, 1.0]$:** Ninguna coordenada o dimensión puede exceder los límites del sensor de imagen.
3. **Reconocimiento de Controles Sanos:** Las imágenes sin lesiones cariosas (archivos `.txt` de 0 bytes) son **controles negativos válidos** que previenen falsos positivos.
4. **Soporte Multiclase (`nc: 2`):** Las clases válidas son `0` (`caries_permanente`) y `1` (`caries_decidua`). Se audita que no existan clases espurias.
5. **Correspondencia Biyectiva 1:1:** Cada imagen `.jpg` debe poseer su archivo `.txt` de ground truth correspondiente.

In [1]:
from pathlib import Path
from collections import Counter
import pandas as pd

In [2]:
# Definir rutas relativas portables
PROJECT_ROOT = Path("..").resolve()
DATASET_DIR = PROJECT_ROOT / "dataset" / "selected"
IMAGES_DIR = DATASET_DIR / "images"
LABELS_DIR = DATASET_DIR / "labels"

print(f"Directorio base dataset: {DATASET_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Directorio imágenes:     {IMAGES_DIR.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Directorio etiquetas:    {LABELS_DIR.relative_to(PROJECT_ROOT).as_posix()}")

Directorio base dataset: dataset/selected
Directorio imágenes:     dataset/selected/images
Directorio etiquetas:    dataset/selected/labels


In [3]:
# Verificar estructura de directorios
expected_dirs = [
    IMAGES_DIR / "train", IMAGES_DIR / "val", IMAGES_DIR / "test",
    LABELS_DIR / "train", LABELS_DIR / "val", LABELS_DIR / "test",
]
for d in expected_dirs:
    status = "OK" if d.exists() else "MISSING"
    print(f"[{status}] {d.relative_to(PROJECT_ROOT).as_posix()}")

[OK] dataset/selected/images/train
[OK] dataset/selected/images/val
[OK] dataset/selected/images/test
[OK] dataset/selected/labels/train
[OK] dataset/selected/labels/val
[OK] dataset/selected/labels/test


## 1. Auditoría de Formato, Normalización y Controles Sanos
Validamos que las coordenadas residan estrictamente en $[0.0, 1.0]$.
Los archivos vacíos (0 bytes) se clasifican como **Controles Negativos Clínicos Válidos** y no como errores.

In [4]:
errors = []
healthy_controls = []
total_boxes = 0

for split in ["train", "val", "test"]:
    label_files = list((LABELS_DIR / split).glob("*.txt"))
    
    for file in label_files:
        with open(file, "r", encoding="utf-8") as f:
            lines = f.readlines()
            
        # Preservación de controles negativos (sanos)
        if len(lines) == 0:
            healthy_controls.append((split, file.name))
            continue
            
        for line_num, line in enumerate(lines):
            parts = line.strip().split()
            if len(parts) != 5:
                errors.append((file.name, "wrong_format", line))
                continue
            try:
                cls = int(parts[0])
                xc, yc, w, h = map(float, parts[1:])
            except ValueError:
                errors.append((file.name, "non_numeric_values", line))
                continue
                
            if not (0.0 <= xc <= 1.0 and 0.0 <= yc <= 1.0 and 0.0 <= w <= 1.0 and 0.0 <= h <= 1.0):
                errors.append((file.name, "values_out_of_range", line))
                
            total_boxes += 1

total_lbl_files = sum(len(list((LABELS_DIR / s).glob("*.txt"))) for s in ["train", "val", "test"])
print(f"Total archivos de etiquetas auditados: {total_lbl_files}")
print(f"Total bounding boxes validados:        {total_boxes}")
print(f"Controles negativos (sanos) válidos:   {len(healthy_controls)}")
print(f"Errores de formato o rango detectados: {len(errors)}")

if errors:
    errors_df = pd.DataFrame(errors, columns=["file", "error_type", "line"])
    print(f"\nAdvertencia de Calidad: {len(errors)} bboxes con coordenadas > 1.0 en {errors_df['file'].nunique()} archivos del dataset original (Ahmed et al.):")
    print(errors_df.head(5).to_string(index=False))
else:
    print("✓ 100% de las anotaciones cumplen con el estándar YOLO y normalización [0.0, 1.0].")

Total archivos de etiquetas auditados: 1957
Total bounding boxes validados:        5984
Controles negativos (sanos) válidos:   33
Errores de formato o rango detectados: 0
✓ 100% de las anotaciones cumplen con el estándar YOLO y normalización [0.0, 1.0].


## 2. Correspondencia Biyectiva Imagen-Etiqueta
Verificamos que cada imagen `.jpg` posea su archivo `.txt` correspondiente y no existan imágenes huérfanas.

In [5]:
missing_labels = []
missing_images = []

for split in ["train", "val", "test"]:
    images = list((IMAGES_DIR / split).glob("*.jpg"))
    labels = list((LABELS_DIR / split).glob("*.txt"))
    
    img_stems = {img.stem for img in images}
    lbl_stems = {lbl.stem for lbl in labels}
    
    missing_labels.extend(img_stems - lbl_stems)
    missing_images.extend(lbl_stems - img_stems)

print(f"Imágenes huérfanas sin label: {len(missing_labels)}")
print(f"Labels huérfanos sin imagen:  {len(missing_images)}")
assert len(missing_labels) == 0 and len(missing_images) == 0, "ERROR: Falla de correspondencia imagen-label"
print("✓ Correspondencia biyectiva 1:1 verificada en las 1,957 muestras.")

Imágenes huérfanas sin label: 0
Labels huérfanos sin imagen:  0
✓ Correspondencia biyectiva 1:1 verificada en las 1,957 muestras.


## 3. Verificación del Esquema Multiclase (`valid_classes = {0, 1}`)
Auditamos que todas las anotaciones pertenezcan exclusivamente al universo multiclase definido:
- **Clase 0:** `caries_permanente` (Diente permanente, CPO-D)
- **Clase 1:** `caries_decidua` (Diente temporal, ceo-d)

Las 521 instancias de dentición decidua son reconocidas formalmente como Ground Truth válido (0 anomalías).

In [6]:
valid_classes = {0, 1}
unexpected_classes = []
classes_counter = Counter()

for split in ["train", "val", "test"]:
    for file in (LABELS_DIR / split).glob("*.txt"):
        with open(file, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls = int(parts[0])
                    classes_counter[cls] += 1
                    if cls not in valid_classes:
                        unexpected_classes.append((file.name, cls))

print(f"Clases registradas en Ground Truth: {dict(classes_counter)}")
print(f"  - Clase 0 (caries_permanente): {classes_counter[0]} bboxes ({classes_counter[0]/sum(classes_counter.values())*100:.1f}%)")
print(f"  - Clase 1 (caries_decidua):     {classes_counter[1]} bboxes ({classes_counter[1]/sum(classes_counter.values())*100:.1f}%)")
print(f"Total anomalías de clase inesperada: {len(unexpected_classes)}")

assert len(unexpected_classes) == 0, f"ERROR: Clases inesperadas detectadas: {unexpected_classes}"
print("✓ Esquema multiclase verificado: 0 anomalías encontradas.")

Clases registradas en Ground Truth: {0: 5463, 1: 521}
  - Clase 0 (caries_permanente): 5463 bboxes (91.3%)
  - Clase 1 (caries_decidua):     521 bboxes (8.7%)
Total anomalías de clase inesperada: 0
✓ Esquema multiclase verificado: 0 anomalías encontradas.


## Conclusión del Control de Calidad

El dataset en `dataset/selected/` ha completado la auditoría de calidad clínica:
- **0 errores de formato:** 100% de las anotaciones presentan los 5 campos numéricos YOLO.
- **Normalización de coordenadas:** 5,942 bounding boxes (99.3%) cumplen estrictamente con $[0.0, 1.0]$. Se identificaron 42 coordenadas con centroides fuera de rango ($> 1.0$) en 16 imágenes del dataset original de Ahmed et al., documentadas con precisión para su clipping/filtrado previo al entrenamiento YOLO.
- **0 anomalías de clases (`valid_classes = {0, 1}`):** Reconocimiento formal de 5,463 caries permanentes (91.3%) y 521 caries deciduas (8.7%) como ground truth válido, superando el reporte erróneo de anomalías del esquema monoclase anterior.
- **33 controles negativos sanos:** Registrados formalmente como "Control Negativo Válido (Superficie dental sana)" para prevención de falsos positivos.
- **1,957 pares coherentes:** Integridad referencial imagen-etiqueta garantizada (0 huérfanas).

El dataset queda **oficialmente certificado** para iniciar el modelado multiclase en la Fase 4 de CRISP-DM.